In [10]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
# Completed: This import matches the Project One CRUD module file and class name.
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name
# Completed: These credentials connect to the aacuser MongoDB account.

username = "aacuser"
password = "DaisyMilo247$"

# Connect to database via CRUD Module
# The CRUD module handles database access so the dashboard code stays focused on display and interaction.
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
# This check prevents an error if the _id column has already been removed.
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
# Completed: The logo image is loaded from the local Codio project folder.
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))
# Completed: The logo is included in the layout below and linked to the requested SNHU website.


def build_rescue_query(filter_type):
    """
    Build and return the MongoDB query for the selected rescue type.

    The filter criteria are based on the Grazioso Salvare rescue type table:
    breed, sex upon outcome, and training age in weeks.
    """

    if filter_type == 'water':
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "$or": [
                {"breed": {"$regex": "Labrador Retriever", "$options": "i"}},
                {"breed": {"$regex": "Chesapeake Bay Retriever", "$options": "i"}},
                {"breed": {"$regex": "Newfoundland", "$options": "i"}}
            ]
        }

    elif filter_type == 'mountain':
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "$or": [
                {"breed": {"$regex": "German Shepherd", "$options": "i"}},
                {"breed": {"$regex": "Alaskan Malamute", "$options": "i"}},
                {"breed": {"$regex": "Old English Sheepdog", "$options": "i"}},
                {"breed": {"$regex": "Siberian Husky", "$options": "i"}},
                {"breed": {"$regex": "Rottweiler", "$options": "i"}}
            ]
        }

    elif filter_type == 'disaster':
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
            "$or": [
                {"breed": {"$regex": "Doberman Pinscher", "$options": "i"}},
                {"breed": {"$regex": "German Shepherd", "$options": "i"}},
                {"breed": {"$regex": "Golden Retriever", "$options": "i"}},
                {"breed": {"$regex": "Bloodhound", "$options": "i"}},
                {"breed": {"$regex": "Rottweiler", "$options": "i"}}
            ]
        }

    else:
        # Reset returns the dashboard to the original unfiltered state.
        return {}


app.layout = html.Div([

    html.Center(
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(
                    encoded_image.decode()
                ),
                height='150px'
            ),
            href='https://www.snhu.edu',
            target='_blank'
        )
    ),

    html.Center(html.B(html.H1('Grazioso Salvare Rescue Dashboard'))),

    html.Center(
        html.H4('Created by Beau Welchel')
    ),

    html.Hr(),

    html.Div([
        
        #FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
        # Completed: Radio buttons allow users to select one rescue type filter at a time.
        html.H4("Filter by Rescue Type"),

        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={
                'display': 'inline-block',
                'margin-right': '20px'
            }
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict('records'),

        #FIXME: Set up the features for your interactive data table to make it user-friendly for your client
        #If you completed the Module Six Assignment, you can copy in the code you created here 
        # Completed: Sorting, filtering, row selection, pagination, and scrolling improve usability.
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable="single",
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10,

        style_table={
            'overflowX': 'auto',
            'height': '400px',
            'overflowY': 'auto'
        },

        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'width': '150px',
            'maxWidth': '200px',
            'whiteSpace': 'normal'
        },

        style_header={
            'fontWeight': 'bold',
            'backgroundColor': 'rgb(230, 230, 230)'
        }
    ),

    html.Br(),
    html.Hr(),

    #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%'}
            ),

            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%'}
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################


@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)

    # Completed: The selected rescue type is converted into a MongoDB query.
    query = build_rescue_query(filter_type)

    # The CRUD module reads matching documents from MongoDB.
    filtered_data = pd.DataFrame.from_records(db.read(query))

    # Remove MongoDB ObjectId values before sending records to the Dash table.
    if '_id' in filtered_data.columns:
        filtered_data.drop(columns=['_id'], inplace=True)

    # Return updated table records.
    return filtered_data.to_dict('records')


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    #return [
    #    dcc.Graph(            
    #        figure = px.pie(df, names='breed', title='Preferred Animals')
    #    )    
    #]

    # Completed: The pie chart updates based on the table data currently displayed.
    if viewData is None:
        dff = df
    elif len(viewData) == 0:
        return [
            html.H4("No data available for the selected filter.")
        ]
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # Confirm that the breed column exists before creating the chart.
    if 'breed' not in dff.columns:
        return [
            html.H4("Breed data is not available.")
        ]

    # Count the top breeds. Limiting to 10 keeps the pie chart readable.
    breed_counts = dff['breed'].value_counts().nlargest(10).reset_index()
    breed_counts.columns = ['breed', 'count']

    return [
        dcc.Graph(
            figure=px.pie(
                breed_counts,
                names='breed',
                values='count',
                title='Top Breeds by Rescue Filter'
            )
        )
    ]


#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):  

    # Use the full dataframe at initial load if Dash has not created derived data yet.
    if viewData is None:
        dff = df
    elif len(viewData) == 0:
        return [
            html.H4("No map data available for the selected filter.")
        ]
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]

    # Prevent map errors if the selected row is outside the filtered data range.
    if row >= len(dff):
        row = 0

    # These are the latitude and longitude column names in the AAC data.
    lat_col = 'location_lat'
    lon_col = 'location_long'

    # Use named columns when available. Fall back to the original starter-code column positions.
    if lat_col in dff.columns and lon_col in dff.columns:
        latitude = dff.iloc[row][lat_col]
        longitude = dff.iloc[row][lon_col]
    else:
        latitude = dff.iloc[row, 13]
        longitude = dff.iloc[row, 14]

    # Handle missing location values by keeping the map centered on Austin.
    if pd.isna(latitude) or pd.isna(longitude):
        latitude = 30.75
        longitude = -97.48

    # Convert coordinates to float values for dash_leaflet.
    latitude = float(latitude)
    longitude = float(longitude)

    # Get marker details from the selected animal record.
    breed = dff.iloc[row]['breed'] if 'breed' in dff.columns else "Unknown Breed"
    name = dff.iloc[row]['name'] if 'name' in dff.columns else "Unknown Name"

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                # Marker with tool tip and popup
                # Column 13 and 14 define the grid-coordinates for the map
                # Column 4 defines the breed for the animal
                # Column 9 defines the name of the animal
                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(breed),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(name),
                            html.H4("Breed"),
                            html.P(breed)
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()

Dash app running on https://husbandfax-janetjeep-3000.codio.io/proxy/8050/
